In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
import torchvision.utils as vutils
import numpy as np
import os
from medmnist import PathMNIST
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.models import inception_v3
from scipy import linalg

In [8]:
# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
num_epochs = 100
batch_size = 128  # Keep as is unless GPU memory is an issue
lr_d = 0.00005  # Reduced to slow discriminator
lr_g = 0.0002   # Adjusted for balance
z_dim = 100
feature_dim = 64  # For feature matching
g_updates = 2     # Number of generator updates per discriminator update

# Data loading and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

Using device: cuda
Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


In [9]:
# Generator for LS-GAN
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 1, 1, 0, bias=False),  # Changed to output [batch_size, 1, 1, 1]
        )
        self.feature_layer = nn.Sequential(*list(self.model.children())[:-1])  # For feature matching

    def forward(self, x):
        output = self.model(x)
        return output.view(x.size(0), -1)  # Flatten to [batch_size, 1]

    def get_features(self, x):
        features = self.feature_layer(x)  # [batch_size, 512, 2, 2]
        return features.mean(dim=[2, 3])  # Average to [batch_size, 512]

In [10]:
# Evaluation functions (unchanged)
def load_inception_model(device):
    inception_model = inception_v3(weights='Inception_V3_Weights.IMAGENET1K_V1').to(device)
    inception_model.eval()
    return inception_model

def get_inception_activations(images, inception_model, device, batch_size=32):
    activations = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            act = inception_model(batch)
            activations.append(act.cpu().numpy())
    return np.concatenate(activations, axis=0)

def compute_fid(real_images, fake_images, inception_model, device, batch_size=32):
    real_acts = get_inception_activations(real_images, inception_model, device, batch_size)
    fake_acts = get_inception_activations(fake_images, inception_model, device, batch_size)
    mu_real, sigma_real = np.mean(real_acts, axis=0), np.cov(real_acts, rowvar=False)
    mu_fake, sigma_fake = np.mean(fake_acts, axis=0), np.cov(fake_acts, rowvar=False)
    diff = mu_real - mu_fake
    covmean = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def compute_inception_score(images, inception_model, device, batch_size=32, splits=10):
    preds = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            pred = inception_model(batch)
            pred = F.softmax(pred, dim=1).cpu().numpy()
            preds.append(pred)
    preds = np.concatenate(preds, axis=0)
    scores = []
    for i in range(splits):
        part = preds[(i * preds.shape[0] // splits):((i + 1) * preds.shape[0] // splits)]
        kl = part * (np.log(part) - np.log(np.mean(part, axis=0, keepdims=True)))
        kl = np.mean(np.sum(kl, axis=1))
        scores.append(np.exp(kl))
    return np.mean(scores), np.std(scores)

def evaluate(generator, train_loader, device, num_samples=5000, z_dim=100, batch_size=128):
    inception_model = load_inception_model(device)
    generator.eval()
    fake_images = []
    with torch.no_grad():
        for _ in range(num_samples // batch_size):
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake = generator(z)
            fake_images.append(fake.cpu())
    fake_images = torch.cat(fake_images, dim=0)[:num_samples]
    
    real_images = []
    for batch in train_loader:
        real = batch[0]
        real_images.append(real)
        if len(real_images) * batch_size >= num_samples:
            break
    real_images = torch.cat(real_images, dim=0)[:num_samples]
    
    fid_score = compute_fid(real_images, fake_images, inception_model, device, batch_size)
    is_mean, is_std = compute_inception_score(fake_images, inception_model, device, batch_size)
    return fid_score, is_mean, is_std

In [11]:
# Training function with improvements
def train_lsgan():
    # Initialize models
    generator = Generator(z_dim=z_dim).to(device)
    discriminator = Discriminator().to(device)
    
    # Optimizers
    optimizer_d = optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))
    optimizer_g = optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
    
    # Loss function
    criterion = nn.MSELoss()
    
    # TensorBoard writer
    writer = SummaryWriter('runs/lsgan_pathmnist_updated')
    
    # Best model tracking
    best_d_loss = float('inf')
    
    # Fixed noise for visualization
    fixed_z = torch.randn(5, z_dim, 1, 1).to(device)
    
    # Create output directory for images
    os.makedirs('generated_images', exist_ok=True)
    
    # Training loop
    for epoch in range(num_epochs):
        discriminator.train()
        generator.train()
        d_loss_total = 0.0
        g_loss_total = 0.0
        
        for i, (real_images, _) in enumerate(tqdm(train_loader, desc=f"LS-GAN Epoch {epoch+1}/{num_epochs}")):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            # Add small noise to real images
            noise = torch.randn_like(real_images) * 0.05
            real_images_noisy = real_images + noise
            
            # Labels with smoothing
            real_labels = torch.ones(batch_size, 1).to(device) * 0.9
            fake_labels = torch.zeros(batch_size, 1).to(device) + 0.1
            
            # Train Discriminator
            optimizer_d.zero_grad()
            real_output = discriminator(real_images_noisy)
            d_loss_real = criterion(real_output, real_labels)
            
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake_images = generator(z)
            fake_output = discriminator(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)
            
            d_loss = (d_loss_real + d_loss_fake) / 2
            d_loss.backward()
            optimizer_d.step()
            
            # Train Generator (multiple updates)
            g_loss_batch = 0.0
            for _ in range(g_updates):
                optimizer_g.zero_grad()
                fake_images = generator(z)
                fake_output = discriminator(fake_images)
                g_loss = criterion(fake_output, real_labels)
                
                # Feature matching loss
                real_features = discriminator.get_features(real_images)
                fake_features = discriminator.get_features(fake_images)
                fm_loss = criterion(fake_features, real_features.detach())
                g_loss = g_loss + 0.1 * fm_loss
                
                g_loss.backward()
                optimizer_g.step()
                g_loss_batch += g_loss.item() / g_updates
            
            d_loss_total += d_loss.item()
            g_loss_total += g_loss_batch
            
            d_fake_mean = fake_output.mean().item()
        
        # Average losses
        d_loss_avg = d_loss_total / len(train_loader)
        g_loss_avg = g_loss_total / len(train_loader)
        
        # Log to TensorBoard
        writer.add_scalar('Loss/Discriminator', d_loss_avg, epoch)
        writer.add_scalar('Loss/Generator', g_loss_avg, epoch)
        writer.add_scalar('Metrics/D_Fake_Mean', d_fake_mean, epoch)
        
        print(f"LS-GAN Epoch {epoch+1}: d_loss={d_loss_avg:.4f}, g_loss={g_loss_avg:.4f}, d_fake_mean={d_fake_mean:.4f}")
        
        # Save best model
        if d_loss_avg < best_d_loss:
            best_d_loss = d_loss_avg
            torch.save(generator.state_dict(), 'best_generator_lsgan.pth')
            print(f"Saved best LS-GAN generator at epoch {epoch+1}")
        
        # Visualize and save images every 5 epochs
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                fake_images = generator(fixed_z)
                real_images_vis = (real_images[:5] + 1) / 2
                fake_images_vis = (fake_images + 1) / 2
                real_grid = vutils.make_grid(real_images_vis, nrow=5, padding=2, normalize=False)
                fake_grid = vutils.make_grid(fake_images_vis, nrow=5, padding=2, normalize=False)
                writer.add_image('Images/Real', real_grid, epoch)
                writer.add_image('Images/Generated', fake_grid, epoch)
                vutils.save_image(fake_images_vis, f"generated_images/epoch_{epoch+1}.png", nrow=5, normalize=False)
        
        # Evaluate every 10 epochs
        if (epoch + 1) % 10 == 0:
            fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim, batch_size=batch_size)
            writer.add_scalar('Metrics/FID', fid_score, epoch)
            writer.add_scalar('Metrics/Inception Score Mean', is_mean, epoch)
            writer.add_scalar('Metrics/Inception Score Std', is_std, epoch)
            print(f"Epoch {epoch+1} - FID: {fid_score:.2f}, IS: {is_mean:.2f} ± {is_std:.2f}")
    
    # Final evaluation
    print("Training completed. Performing final evaluation...")
    fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim, batch_size=batch_size)
    print(f"Final FID Score: {fid_score:.2f}")
    print(f"Final Inception Score: {is_mean:.2f} ± {is_std:.2f}")
    
    writer.add_scalar('Metrics/Final FID', fid_score, num_epochs)
    writer.add_scalar('Metrics/Final Inception Score Mean', is_mean, num_epochs)
    writer.add_scalar('Metrics/Final Inception Score Std', is_std, num_epochs)
    writer.close()

In [12]:
# Run the training
if __name__ == "__main__":
    train_lsgan()

LS-GAN Epoch 1/100:   0%|          | 0/704 [00:00<?, ?it/s]c:\Anaconda3\envs\torch-env\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([128, 1])) that is different to the input size (torch.Size([128, 16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
LS-GAN Epoch 1/100: 100%|█████████▉| 702/704 [01:22<00:00,  9.85it/s]c:\Anaconda3\envs\torch-env\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([12, 1])) that is different to the input size (torch.Size([12, 16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
LS-GAN Epoch 1/100: 100%|██████████| 704/704 [01:23<00:00,  8.47it/s]


LS-GAN Epoch 1: d_loss=0.0923, g_loss=0.5087, d_fake_mean=0.1926
Saved best LS-GAN generator at epoch 1


LS-GAN Epoch 2/100: 100%|██████████| 704/704 [01:28<00:00,  7.92it/s]


LS-GAN Epoch 2: d_loss=0.0278, g_loss=0.5966, d_fake_mean=-0.0152
Saved best LS-GAN generator at epoch 2


LS-GAN Epoch 3/100: 100%|██████████| 704/704 [01:30<00:00,  7.78it/s]


...

LS-GAN Epoch 98: d_loss=0.0001, g_loss=0.6768, d_fake_mean=0.1010


LS-GAN Epoch 99/100: 100%|██████████| 704/704 [01:26<00:00,  8.18it/s]


LS-GAN Epoch 99: d_loss=0.0001, g_loss=0.6770, d_fake_mean=0.1257


LS-GAN Epoch 100/100: 100%|██████████| 704/704 [01:26<00:00,  8.17it/s]


LS-GAN Epoch 100: d_loss=0.0001, g_loss=0.6769, d_fake_mean=0.1041
Epoch 100 - FID: 1097.52, IS: 1.00 ± 0.00
Training completed. Performing final evaluation...
Final FID Score: 1093.33
Final Inception Score: 1.00 ± 0.00
